## Phase 2.1: Data Engineering & Structuring
### Environment & Architecture Setup
To ensure strict reproducibility across the research team's local machines, we establish absolute architectural paths relative to this notebook. This cell initializes the computational environment and guarantees the target directories (`02_intermediate` and `03_processed`) exist before data pipelines are executed.

In [15]:
import pandas as pd
import numpy as np
import os

# 1. Establish strict architectural paths
RAW_DIR = "../data/01_raw/"
INTERMEDIATE_DIR = "../data/02_intermediate/"
PROCESSED_DIR = "../data/03_processed/"

os.makedirs(INTERMEDIATE_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
print("Environment architecture validated.")

Environment architecture validated.


### Comprehensive Data Ingestion
We systematically load all raw data required to test our four core hypotheses. This includes macroeconomic structural variables (World Bank), demand proxies (Google Trends), liquidity and network usage (CoinMetrics, yfinance), and low-level infrastructural network costs (Etherscan, Tronscan).

In [16]:
# 2. Ingest Macro & Crypto Demand Data (H1, H2)
wb_df = pd.read_csv(f"{RAW_DIR}worldbank/all_indicators.csv")
cm_add_df = pd.read_csv(f"{RAW_DIR}coinmetrics/active_addresses.csv")
gt_df = pd.read_csv(f"{RAW_DIR}googletrends/global_search_intensity.csv")
yf_usdc_df = pd.read_csv(f"{RAW_DIR}yfinance/USDC_daily_volume.csv")
yf_usdt_df = pd.read_csv(f"{RAW_DIR}yfinance/USDT_daily_volume.csv")

# 3. Ingest Infrastructure Cost Data (H3, H4)
eth_tx_df = pd.read_csv(f"{RAW_DIR}etherscan/usdc_transfers_sample.csv")
tron_tx_df = pd.read_csv(f"{RAW_DIR}tronscan/usdt_transfers_sample.csv")
eth_price_df = pd.read_csv(f"{RAW_DIR}coinmetrics/eth_trx_price_usd.csv")

print("All raw datasets (Macro, Demand, and Infrastructure) loaded successfully.")

All raw datasets (Macro, Demand, and Infrastructure) loaded successfully.


### Universal Temporal Standardization & Structural Break Injection
Financial APIs and macroeconomic databases format time inconsistently (e.g., Etherscan uses 10-digit UNIX seconds, Tronscan uses 13-digit UNIX milliseconds, World Bank uses annual integers). This function dynamically detects and standardizes all temporal data to a uniform `YYYY-MM-DD` format. 

Furthermore, it strictly truncates all datasets to our defined 2020-2025 timeframe and injects the `post_nov_2022` dummy variable to enable pre/post structural break analysis centered around the FTX liquidity crisis.

In [17]:
def standardize_dates(df, is_annual=False):
    """
    Dynamically identifies temporal columns and robustly handles UNIX seconds 
    (Etherscan) vs UNIX milliseconds (Tronscan) vs standard strings.
    """
    possible_names = ['date', 'Date', 'time', 'timestamp', 'TimeStamp', 'block_timestamp', 'year', 'Year']
    time_col = next((col for col in possible_names if col in df.columns), None)
    
    if not time_col:
        raise KeyError(f"Critical Error: No temporal column found. Available columns: {df.columns}")

    if is_annual:
        df['date'] = pd.to_datetime(df[time_col].astype(str), format='%Y')
        if time_col != 'date':
            df = df.drop(columns=[time_col])
    else:
        # Check if the column is numeric (UNIX timestamps)
        if pd.api.types.is_numeric_dtype(df[time_col]):
            # 10-digit UNIX (Seconds - e.g., Etherscan)
            if df[time_col].max() < 20000000000: 
                df['date'] = pd.to_datetime(df[time_col], unit='s', errors='coerce')
            # 13-digit UNIX (Milliseconds - e.g., Tronscan)
            else:
                df['date'] = pd.to_datetime(df[time_col], unit='ms', errors='coerce')
        else:
            # Standard string dates
            df['date'] = pd.to_datetime(df[time_col], errors='coerce')

    if df['date'].dt.tz is not None:
        df['date'] = df['date'].dt.tz_localize(None)
    df['date'] = df['date'].dt.normalize()
    
    # 2020-2025 Truncation & Structural Break Dummy
    mask = (df['date'] >= '2020-01-01') & (df['date'] <= '2025-12-31')
    df = df.loc[mask].copy()
    df['post_nov_2022'] = (df['date'] > '2022-11-30').astype(int)
    
    if time_col != 'date' and time_col in df.columns:
         df = df.drop(columns=[time_col])
            
    return df.sort_values(by=['date']).reset_index(drop=True)

# Apply bulletproof standardizer
wb_df = standardize_dates(wb_df, is_annual=True)
cm_add_df = standardize_dates(cm_add_df)
gt_df = standardize_dates(gt_df)
yf_usdc_df = standardize_dates(yf_usdc_df)
yf_usdt_df = standardize_dates(yf_usdt_df)
eth_tx_df = standardize_dates(eth_tx_df)
tron_tx_df = standardize_dates(tron_tx_df)
eth_price_df = standardize_dates(eth_price_df)

print("Temporal standardization and 'post_nov_2022' dummy injected across ALL datasets.")

Temporal standardization and 'post_nov_2022' dummy injected across ALL datasets.


### Macroeconomic Imputation & The "Namibia" Correction
Macroeconomic variables such as inflation and banking penetration inherently suffer from reporting lags. Dropping these rows introduces severe truncation bias. 

First, we resolve a known pandas ingestion anomaly where Namibia's ISO code ('NA') is misread as a Null value. We then apply strict linear interpolation, grouped by sovereign country codes, to mathematically estimate the 17 missing macro reporting gaps without bleeding data across borders.

In [18]:
# Fix Pandas "Namibia" bug ('NA' read as Null)
if 'country_name' in wb_df.columns:
    wb_df.loc[wb_df['country_name'] == 'Namibia', 'country_code'] = 'NAM'
    wb_df['country_code'] = wb_df.groupby('country_name')['country_code'].transform(lambda x: x.ffill().bfill())
    wb_df['country_code'] = wb_df['country_code'].fillna('UNKNOWN')

# Isolate numeric columns for strict mathematical interpolation
numeric_cols = [c for c in wb_df.select_dtypes(include=[np.number]).columns if c not in ['post_nov_2022']]

# Interpolate missing trends
if 'country_code' in wb_df.columns:
    wb_df[numeric_cols] = wb_df.groupby('country_code')[numeric_cols].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )
wb_df[numeric_cols] = wb_df[numeric_cols].ffill().bfill()

assert wb_df.isna().sum().sum() == 0, "Nulls persist in Macro data."
print("World Bank macroeconomic data fully imputed and validated.")

World Bank macroeconomic data fully imputed and validated.


### Infrastructure Outlier Filtering & USD Normalization (Hypothesis 4)
To accurately compare the empirical "cost friction" of blockchain networks against legacy correspondent banking (SWIFT), we must normalize raw protocol fees (Gwei) into USD using historical daily prices. 

Crucially, we filter out the top 5% of Ethereum transaction fees. This isolates the baseline utility cost of stablecoin transfers by removing extreme outliers caused by speculative network congestion (e.g., NFT mint gas wars), which would otherwise artificially skew our H4 cost regressions.

In [19]:
# 1. USD Normalization: Merge daily ETH price to calculate exact USD cost
eth_tx_df = eth_tx_df.merge(eth_price_df[['date', 'PriceUSD']], on='date', how='left')

# Find the gas fee column dynamically (handles 'gas_fee_eth', 'fee', 'gas_fee')
gas_col = next((col for col in eth_tx_df.columns if 'fee' in col.lower()), None)

if gas_col and 'PriceUSD' in eth_tx_df.columns:
    eth_tx_df['fee_usd'] = eth_tx_df[gas_col] * eth_tx_df['PriceUSD']
    
    # 2. Outlier Filtering: Remove top 5% gas spikes (NFT mint congestion)
    initial_count = len(eth_tx_df)
    gas_cap_95th = eth_tx_df['fee_usd'].quantile(0.95)
    eth_tx_df = eth_tx_df[eth_tx_df['fee_usd'] <= gas_cap_95th].copy()
    
    print(f"Filtered {initial_count - len(eth_tx_df)} extreme ETH gas outliers (> ${gas_cap_95th:.2f}).")
else:
    print("Warning: Could not dynamically calculate fee_usd. Check Etherscan column structures.")

Filtered 7200 extreme ETH gas outliers (> $47.45).


### Master Export to Intermediate Directory
With all temporal misalignments, null values, and infrastructural outliers completely resolved, we export these 7 atomic datasets to the `02_intermediate` directory. This concludes Phase 2.1.

In [20]:
# Save all strictly clean, atomic datasets
wb_df.to_csv(f"{INTERMEDIATE_DIR}wb_cleaned.csv", index=False)
cm_add_df.to_csv(f"{INTERMEDIATE_DIR}cm_active_addresses_cleaned.csv", index=False)
gt_df.to_csv(f"{INTERMEDIATE_DIR}gt_global_intensity_cleaned.csv", index=False)
yf_usdc_df.to_csv(f"{INTERMEDIATE_DIR}yf_usdc_volume_cleaned.csv", index=False)
yf_usdt_df.to_csv(f"{INTERMEDIATE_DIR}yf_usdt_volume_cleaned.csv", index=False)
eth_tx_df.to_csv(f"{INTERMEDIATE_DIR}etherscan_usdc_cleaned.csv", index=False)
tron_tx_df.to_csv(f"{INTERMEDIATE_DIR}tronscan_usdt_cleaned.csv", index=False)

print("PHASE 2.1 OFFICIALLY COMPLETE. All 7 datasets safely exported to /02_intermediate/")

PHASE 2.1 OFFICIALLY COMPLETE. All 7 datasets safely exported to /02_intermediate/
